# **SETUP & LIBRARIES**

In [1]:
import pandas as pd
import os
import sys

# allow src imports
sys.path.append("../..")

from src.data_loader import load_dataset
from src.feature_engineering import convert_time, compute_time_diff
from src.metrics import summarize_times, format_time

# load germany dataset
data = load_dataset("berlin")

stops = data["stops"]
stop_times = data["stop_times"]
trips = data["trips"]
routes = data["routes"]

stops.head(10)

,stop_id,stop_code,stop_name,stop_desc,stop_lat,stop_lon,location_type,parent_station,wheelchair_boarding,platform_code,zone_id,level_id
0,de:11000:900007104::2,5,S Nordbahnhof (Berlin),Tram u. Bussteig Invalidenstraße ht. Gartenstraße,52.531683,13.388813,0,NaN,0,NaN,5555 S Nordbahnhof (Berlin),4.0
1,de:11000:900100007::3,NaN,S Oranienburger Str. (Berlin),Ersatzhalt Tucholskystraße vor Oranienburger S...,52.524724,13.392833,0,NaN,0,NaN,5555 S Oranienburger Str. (Berlin),4.0
2,de:12070:900215110:1:50,NaN,"Bad Wilsnack, Bahnhof",Bahnsteig Gleis 2,52.960114,11.949402,0,de:12070:900215110,1,2,"4533 Bad Wilsnack, Bahnhof",50.0
3,de:12070:900215110:2:51,NaN,"Bad Wilsnack, Bahnhof",Bahnsteig Gleis 3,52.960219,11.949528,0,de:12070:900215110,1,3,"4533 Bad Wilsnack, Bahnhof",50.0
4,de:12062:900415465:1:50,NaN,"Prösen, Bahnhof",Bahnsteig Gleis 1,51.434919,13.488216,0,de:12062:900415465,1,1,"7958 Prösen, Bahnhof",4.0
5,de:12062:900415465:2:51,NaN,"Prösen, Bahnhof",Bahnsteig Gleis 2,51.434886,13.488277,0,de:12062:900415465,1,2,"7958 Prösen, Bahnhof",4.0
6,de:12072:900245025:1:50,NaN,"Rangsdorf, Bahnhof",Bahnsteig Gleis 1,52.294683,13.430835,0,de:12072:900245025,1,1,"6056 Rangsdorf, Bahnhof",4.0
7,de:12072:900245025:2:51,NaN,"Rangsdorf, Bahnhof",Bahnsteig Gleis 2,52.294338,13.430585,0,de:12072:900245025,1,2,"6056 Rangsdorf, Bahnhof",4.0
8,de:14713:8010205,NaN,"Leipzig, Hauptbahnhof",NaN,51.345471,12.382064,0,NaN,0,NaN,NaN,NaN
9,de:12066:900435000:1:50,NaN,"Senftenberg, Bahnhof",Bahnsteig 1,51.527237,14.004368,0,de:12066:900435000,1,1,"7765 Senftenberg, Bahnhof",1.0


# **BASIC DATA OVERVIEW**

In [2]:
# inspect stops
print("\nStop Columns:")
for col in stops.columns:
    print(f"- {col}")


Stop Columns:
- stop_id
- stop_code
- stop_name
- stop_desc
- stop_lat
- stop_lon
- location_type
- parent_station
- wheelchair_boarding
- platform_code
- zone_id
- level_id


In [3]:
# inspect stop_times
print("\nStop Times Columns:")
for col in stop_times.columns:
    print(f"- {col}")


Stop Times Columns:
- trip_id
- stop_id
- stop_sequence
- pickup_type
- drop_off_type
- stop_headsign
- arrival_time
- departure_time


In [4]:
print("Total Stops:", stops.shape[0])
print("Total Stop Times:", stop_times.shape[0])
print("Total Trips:", trips.shape[0])
print("Total Routes:", routes.shape[0])

Total Stops: 41931
Total Stop Times: 5939398
Total Trips: 262828
Total Routes: 1309


# **FEATURE ENGINEERING**

In [5]:
# convert time → timedelta
stop_times = convert_time(stop_times)

# compute inter-stop travel time
stop_times = compute_time_diff(stop_times)

# create previous stop reference
stop_times["prev_stop_id"] = stop_times.groupby("trip_id")["stop_id"].shift(1)

# light filtering (DO NOT over-filter early)
stop_times = stop_times[
    (stop_times["time_diff"] > pd.Timedelta(seconds=20)) &
    (stop_times["time_diff"] <= pd.Timedelta(minutes=90))
].copy()

# sanity check
print("Remaining segments:", len(stop_times))

Remaining segments: 5665344


# **SYSTEM TRAVEL TIME SUMMARY**

In [6]:
# summary stats
summary = summarize_times(stop_times["time_diff"])
print(summary)

{'count': 5665344, 'mean': Timedelta('0 days 00:01:35.129643319'), 'min': Timedelta('0 days 00:00:24'), '25%': Timedelta('0 days 00:01:00'), '50%': Timedelta('0 days 00:01:00'), '75%': Timedelta('0 days 00:02:00'), 'max': Timedelta('0 days 01:24:24')}


# **DATA MERGING**

In [7]:
# merge trips
final_df = stop_times.merge(
    trips[["trip_id", "route_id"]],
    on="trip_id",
    how="left"
)

# merge routes
final_df = final_df.merge(
    routes[["route_id", "route_short_name", "route_type"]],
    on="route_id",
    how="left"
)

# merge stop names
final_df = final_df.merge(
    stops[["stop_id", "stop_name"]],
    on="stop_id",
    how="left"
)

# merge previous stop names
final_df = final_df.merge(
    stops[["stop_id", "stop_name"]].rename(
        columns={"stop_id": "prev_stop_id", "stop_name": "prev_stop_name"}
    ),
    on="prev_stop_id",
    how="left"
)

print("Merged dataset shape:", final_df.shape)

Merged dataset shape: (5665344, 16)


# **TRANSPORT MODE MAPPING**

In [8]:
# GTFS + extended route types
route_type_map = {
    0: "Tram",
    1: "Subway",
    2: "Rail",
    3: "Bus",
    100: "Rail",
    109: "Rail",     # S-Bahn FIX
    400: "Subway",
    700: "Bus"
}

final_df["route_type_name"] = final_df["route_type"].map(route_type_map)

# check unknowns BEFORE assigning
unknown_mask = final_df["route_type_name"].isna()
print("Unknown route types:", unknown_mask.sum())

# assign unknown
final_df.loc[unknown_mask, "route_type_name"] = "Other"

print("\nMode Distribution:")
print(final_df["route_type_name"].value_counts())

Unknown route types: 912535

Mode Distribution:
route_type_name
Bus       4178325
Other      912535
Rail       319359
Subway     255125
Name: count, dtype: int64


# **RAIL CLASSIFICATION**

In [9]:
def classify_rail(row):
    route = str(row["route_short_name"]).upper().strip()

    if row["route_type_name"] == "Rail":

        # S-Bahn detection
        if route.startswith("S") and any(char.isdigit() for char in route):
            return "S-Bahn"

        elif route.startswith(("RE", "RB")):
            return "Regional"

        elif route.startswith(("ICE", "IC", "EC")):
            return "Long-Distance"

        else:
            return "Other Rail"

    return None

final_df["rail_type"] = final_df.apply(classify_rail, axis=1)

print("\nRail Breakdown:")
print(final_df["rail_type"].value_counts())


Rail Breakdown:
rail_type
S-Bahn        263182
Regional       55242
Other Rail       935
Name: count, dtype: int64


# **BERLIN FILTER**

In [10]:
# Berlin bounding box
berlin_stops = stops[
    (stops["stop_lat"].between(52.3, 52.7)) &
    (stops["stop_lon"].between(13.0, 13.8))
]

berlin_stop_ids = set(berlin_stops["stop_id"])

# keep only segments fully inside Berlin
final_df = final_df[
    (final_df["stop_id"].isin(berlin_stop_ids)) &
    (final_df["prev_stop_id"].isin(berlin_stop_ids))
].copy()

print("Filtered Berlin dataset:", final_df.shape)

Filtered Berlin dataset: (5083619, 18)


# **URBAN DATASET CREATION**

In [11]:
urban_df = final_df[
    (final_df["route_type_name"].isin(["Bus", "Tram", "Subway"])) |
    (final_df["rail_type"] == "S-Bahn")
].copy()

# realistic urban travel times
urban_df = urban_df[
    (urban_df["time_diff"] >= pd.Timedelta(seconds=20)) &
    (urban_df["time_diff"] <= pd.Timedelta(minutes=60))
]

print("\nUrban Mode Distribution:")
print(urban_df["route_type_name"].value_counts())

print("\nUrban Rail Breakdown:")
print(urban_df["rail_type"].value_counts())


Urban Mode Distribution:
route_type_name
Bus       3687428
Subway     255125
Rail       252805
Name: count, dtype: int64

Urban Rail Breakdown:
rail_type
S-Bahn    252805
Name: count, dtype: int64


# **SEGMENT CREATION**

In [12]:
# remove invalid rows (important before creating segments)
urban_df = urban_df[
    urban_df["prev_stop_name"].notna() &
    urban_df["stop_name"].notna()
]

# create directional segment (A → B)
urban_df["segment_id"] = (
    urban_df["prev_stop_name"] + " → " + urban_df["stop_name"]
)

# **AGGREGATION (SEGMENT LEVEL)**

In [13]:
# ensure rail_type exists for grouping (avoid NaN issues)
urban_df["rail_type"] = urban_df["rail_type"].fillna("Non-Rail")

# aggregate to segment level
urban_dashboard = urban_df.groupby(
    ["route_short_name", "route_type_name", "rail_type", "segment_id"]
).agg(
    avg_travel_time=("time_diff", "mean"),
    trip_count=("time_diff", "count")
).reset_index()

In [14]:
# split segments into stops
split = urban_dashboard["segment_id"].str.split(" → ", expand=True)

urban_dashboard["from_stop"] = split[0]
urban_dashboard["to_stop"] = split[1]

# readable time format
urban_dashboard["avg_time_readable"] = (
    urban_dashboard["avg_travel_time"].apply(format_time)
)

The history saving thread hit an unexpected error (OperationalError('database is locked')).History will not be written to the database.


# **MODE STANDARDIZATION**

In [15]:
# base mode
urban_dashboard["mode"] = urban_dashboard["route_type_name"]

# override Rail → S-Bahn
urban_dashboard.loc[
    urban_dashboard["rail_type"] == "S-Bahn", "mode"
] = "S-Bahn"

# rename Subway → U-Bahn (Berlin naming)
urban_dashboard.loc[
    urban_dashboard["mode"] == "Subway", "mode"
] = "U-Bahn"

# REMOVE noise modes
urban_dashboard = urban_dashboard[
    urban_dashboard["mode"].isin(["Bus", "U-Bahn", "S-Bahn", "Tram"])
]

# # **FINAL VALIDATION CHECKS**

In [16]:
print("\nFinal Mode Breakdown (Segments):")
print(urban_dashboard["mode"].value_counts())

print("\nFinal Mode Breakdown (Trips):")
print(urban_dashboard.groupby("mode")["trip_count"].sum())


Final Mode Breakdown (Segments):
mode
Bus       17430
S-Bahn      862
U-Bahn      392
Name: count, dtype: int64

Final Mode Breakdown (Trips):
mode
Bus       3687428
S-Bahn     252805
U-Bahn     255125
Name: trip_count, dtype: int64


# **EXPORT**

In [17]:
urban_dashboard.to_csv("../../data/processed/berlin/urban_dashboard.csv", index=False)

print("\nBerlin urban dataset saved successfully")


Berlin urban dataset saved successfully
